# Week 09B — Training and Deploying a Custom Detector

**MCTA 4364 Machine Vision** · Session 2 of Week 9 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W09_data_centric/W09B_train_deploy_detector.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W09_data_centric/W09B_train_deploy_detector.ipynb)

In Week 8B a COCO detector and a zero-shot detector both failed on our conveyor parts. Today you build the dependable solution end to end:
**create a labelled dataset**, **fine-tune** YOLO on it, **evaluate** it honestly (including under a camera change), **export** it to ONNX, run it
**without PyTorch** (ONNX Runtime and OpenCV), and see the same model called from **C#**. This is the path from notebook to production line.

### Learning outcomes
By the end of this session you will be able to:
1. Organise a detection dataset in **YOLO format** (images, label files, `data.yaml`) and check it.
2. **Fine-tune** a pretrained detector, choose the key hyper-parameters for your hardware, and read the training curves.
3. Evaluate with mAP, per-class AP, a confusion matrix and a **shifted test set**.
4. **Export** to ONNX and run the model with ONNX Runtime and OpenCV DNN, with your own pre- and post-processing.
5. Size a deployment: **latency budget**, required frame rate for a conveyor, and the runtime options (CPU, GPU, C#, edge).

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–10 | 1–2 | Plan; build and check the dataset |
| 10–30 | 3 | Fine-tune YOLO11n (runs while we discuss hyper-parameters and augmentation) |
| 30–45 | 4 | Evaluate: mAP, confusion, failure gallery, shifted camera |
| 45–62 | 5 | Export to ONNX; ONNX Runtime and OpenCV DNN; speed |
| 62–72 | 6–7 | Line-rate engineering; the C# corner |
| 72–80 | 10 | Quiz and exit ticket |
| Home | 8–9 | Exercises with self-checks, challenge |

**Hardware:** training adapts to your machine: about 4 minutes on a laptop CPU (small run), 3–5 minutes on a 4 GB GPU (full run).

> **Licence.** Ultralytics YOLO is AGPL-3.0 (see Week 8B). The deployment part (ONNX, ONNX Runtime, OpenCV) works the same for permissively licensed
> detectors such as RT-DETR / D-FINE exported from `transformers` or torchvision models.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly",
          "ultralytics": "ultralytics", "onnx": "onnx", "onnxruntime": "onnxruntime"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import shutil
import time
from pathlib import Path

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import csv
import torch
import torchvision
from ultralytics import YOLO

import partsim
from cvhelpers import SMOKE, check, data_dir, interact, pipeline_diagram, quiz, show
from dlhelpers import hw_profile, set_seed

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEVICE = 0 if HW["cuda"] else "cpu"
set_seed(0)
NAMES = partsim.SCENE_CLASSES
ROOT = data_dir("w09b")
WEIGHTS = data_dir("weights")
if SMOKE:
    N_TRAIN, N_VAL, EPOCHS, IMGSZ, BATCH = 16, 8, 1, 320, 8
elif HW["cuda"]:
    N_TRAIN, N_VAL, EPOCHS, IMGSZ, BATCH = 400, 80, 40, 640, int(16 * HW["batch_scale"])
else:
    N_TRAIN, N_VAL, EPOCHS, IMGSZ, BATCH = 200, 50, 10, 480, 16
print(f"{N_TRAIN} training / {N_VAL} validation scenes | {EPOCHS} epochs at {IMGSZ} px | batch {BATCH} | device {DEVICE}")

## 1. The plan

In [ ]:
fig, ax = plt.subplots(figsize=(22, 2.6))
pipeline_diagram(["scenes +\nlabels", "YOLO dataset\n(images, .txt, yaml)", "fine-tune\n(COCO-pretrained)", "validate\n(mAP, confusion, shift)",
                  "export\nONNX", "ONNX Runtime /\nOpenCV DNN (Python)", "C# / edge\n(same .onnx)"],
                 colors=["#f1f3f4", "#e8f0fe", "#fef7e0", "#fce8e6", "#e6f4ea", "#e6f4ea", "#e6f4ea"],
                 notes=["partsim", "Section 2", "Section 3", "Section 4", "Section 5", "Section 5", "Section 7"], ax=ax, fontsize=10)
ax.set_title("Figure 1.1 - From images to a deployed detector", fontweight="bold")
plt.show()

## 2. Building the dataset

A YOLO dataset is a folder of images and a parallel folder of **label files** (one `.txt` per image, one line `class cx cy w h` per object,
normalised to 0–1), plus a `data.yaml` that lists the splits and the class names.

We apply Week 9A's lessons while creating it:
- **coverage**: the training scenes use a random exposure between 0.6 and 1.1, so the model sees darker and brighter images;
- **no leakage**: training, validation and test scenes come from different random seeds ("different days");
- a separate **shifted test set** (a darker, noisier camera) measures robustness honestly.

In [ ]:
def yolo_lines_ref(boxes, labels, w, h):
    return [f"{int(l)} {(b[0] + b[2]) / 2 / w:.6f} {(b[1] + b[3]) / 2 / h:.6f} {(b[2] - b[0]) / w:.6f} {(b[3] - b[1]) / h:.6f}"
            for b, l in zip(boxes, labels)]


def write_split(split, n, seed, exposure=(1.0, 1.0), noise=3.0):
    (ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    for i in range(n):
        img, b, l = partsim.make_scene(rng.integers(1 << 31), exposure=rng.uniform(*exposure), noise=noise)
        cv2.imwrite(str(ROOT / "images" / split / f"{split}_{i:04d}.jpg"), img)
        (ROOT / "labels" / split / f"{split}_{i:04d}.txt").write_text("\n".join(yolo_lines_ref(b, l, img.shape[1], img.shape[0])) + "\n")


t0 = time.perf_counter()
shutil.rmtree(ROOT / "images", ignore_errors=True); shutil.rmtree(ROOT / "labels", ignore_errors=True)
write_split("train", N_TRAIN, seed=100, exposure=(0.6, 1.1))
write_split("val", N_VAL, seed=200)
write_split("test_dark", N_VAL, seed=300, exposure=(0.5, 0.5), noise=6.0)
yaml_text = f"path: {ROOT.resolve()}\ntrain: images/train\nval: images/val\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(NAMES))
(ROOT / "data.yaml").write_text(yaml_text)
(ROOT / "data_dark.yaml").write_text(yaml_text.replace("val: images/val", "val: images/test_dark"))
print(f"dataset written in {time.perf_counter() - t0:.0f} s\n")
print("data.yaml:\n" + yaml_text)
first_label = sorted((ROOT / "labels" / "train").glob("*.txt"))[0]
print(f"{first_label.name}:\n" + first_label.read_text())

In [ ]:
def read_yolo(label_path, w, h):
    rows = np.loadtxt(label_path, ndmin=2)
    cls, cx, cy, bw, bh = rows.T
    return np.stack([(cx - bw / 2) * w, (cy - bh / 2) * h, (cx + bw / 2) * w, (cy + bh / 2) * h], 1), cls.astype(int)


fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))
for ax, split in zip(axes, ["train", "val", "test_dark"]):
    img_path = sorted((ROOT / "images" / split).glob("*.jpg"))[1]
    img = cv2.imread(str(img_path))
    b, l = read_yolo(ROOT / "labels" / split / (img_path.stem + ".txt"), img.shape[1], img.shape[0])
    ax.imshow(partsim.draw_boxes(img, b, l)[..., ::-1]); ax.axis("off"); ax.set_title(f"{split}: {img_path.name} (labels read back from disk)")
plt.suptitle("Figure 2.1 - Always draw the label files back on the images before training", fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Fine-tuning YOLO11n

We start from **COCO-pretrained** weights (transfer learning, Week 6B): the backbone already knows edges, shapes and textures; only the head
must learn our four classes. The most important settings:

| argument | meaning | our choice |
|---|---|---|
| `epochs` | passes over the data | 10 on CPU, 40 on GPU |
| `imgsz` | training/inference size (multiple of 32) | 480 on CPU, 640 on GPU |
| `batch` | images per step; lower it if the GPU runs out of memory | 16 (4 GB GPU: 8–16) |
| `lr0`, `optimizer` | learning rate and optimiser | `auto` (the defaults are good) |
| `mosaic`, `hsv_v`, `fliplr`, `flipud` | augmentation: 4-image mosaics, brightness, flips | defaults + `flipud=0.5` (a part upside down is still the same part) |
| `degrees` | rotation augmentation | **0**: rotating an axis-aligned box makes it loose (its corners stick out), which hurts box accuracy; the data already contain every orientation |
| `patience` | early stopping | default |
| `amp` | mixed precision on GPU | on |

Training writes everything (curves, plots, `best.pt`) into a run folder.

In [ ]:
model = YOLO(str(WEIGHTS / "yolo11n.pt"))
t0 = time.perf_counter()
train_results = model.train(data=str(ROOT / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE, workers=HW["num_workers"],
                            flipud=0.5, project=str(ROOT / "runs"), name="conveyor", exist_ok=True, plots=True, verbose=False,
                            seed=0, deterministic=True, amp=HW["amp"])
train_time = time.perf_counter() - t0
RUN = Path(model.trainer.save_dir)
print(f"trained in {train_time / 60:.1f} min; results in {RUN}")

In [ ]:
with open(RUN / "results.csv") as f:
    rows = list(csv.DictReader(f))
log = {k.strip(): np.array([float(r[k]) for r in rows]) for k in rows[0]}
fig, axes = plt.subplots(1, 3, figsize=(20, 4.2))
for k in ["train/box_loss", "train/cls_loss", "train/dfl_loss"]:
    axes[0].plot(log["epoch"], log[k], "o-", label=k.split("/")[1])
axes[0].set_title("training losses"); axes[0].set_xlabel("epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)
for k in ["val/box_loss", "val/cls_loss", "val/dfl_loss"]:
    axes[1].plot(log["epoch"], log[k], "o-", label=k.split("/")[1])
axes[1].set_title("validation losses"); axes[1].set_xlabel("epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)
for k, lab in [("metrics/mAP50(B)", "mAP50"), ("metrics/mAP50-95(B)", "mAP50-95"), ("metrics/precision(B)", "precision"), ("metrics/recall(B)", "recall")]:
    axes[2].plot(log["epoch"], log[k], "o-", label=lab)
axes[2].set_ylim(0, 1.02); axes[2].set_title("validation metrics"); axes[2].set_xlabel("epoch"); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.suptitle("Figure 3.1 - Training curves from results.csv", fontweight="bold")
plt.tight_layout(); plt.show()
batch_img = RUN / "train_batch0.jpg"
if batch_img.exists():
    show(cv2.imread(str(batch_img)), titles=["train_batch0.jpg: what the network actually saw (mosaic, flips, brightness and scale changes)"], figsize=(12, 12))

## 4. Honest evaluation

We load the **best** checkpoint and evaluate it on the validation scenes and on the **darker, noisier camera** that was never used for training
or model selection.

In [ ]:
best = YOLO(str(RUN / "weights" / "best.pt"))
m_val = best.val(data=str(ROOT / "data.yaml"), imgsz=IMGSZ, device=DEVICE, plots=True, verbose=False,     # plots=True fills the confusion matrix
                 project=str(ROOT / "runs"), name="val", exist_ok=True)
m_dark = best.val(data=str(ROOT / "data_dark.yaml"), imgsz=IMGSZ, device=DEVICE, plots=True, verbose=False,
                  project=str(ROOT / "runs"), name="val_dark", exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(21, 4.8), gridspec_kw=dict(width_ratios=[1.2, 1, 1]))
xs = np.arange(len(NAMES))
ap_val = np.zeros(len(NAMES)); ap_val[m_val.box.ap_class_index] = m_val.box.ap
ap_dark = np.zeros(len(NAMES)); ap_dark[m_dark.box.ap_class_index] = m_dark.box.ap
axes[0].bar(xs - 0.2, ap_val, 0.4, label=f"validation (mAP50-95 {m_val.box.map:.3f})", color="#1a73e8")
axes[0].bar(xs + 0.2, ap_dark, 0.4, label=f"dark camera (mAP50-95 {m_dark.box.map:.3f})", color="#f29900")
axes[0].set_xticks(xs); axes[0].set_xticklabels(NAMES); axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("AP50-95"); axes[0].legend(loc="lower left")
axes[0].grid(alpha=0.3, axis="y"); axes[0].set_title("per-class AP")
for ax, m, name in [(axes[1], m_val, "validation"), (axes[2], m_dark, "dark camera")]:
    cm = m.confusion_matrix.matrix                                   # (C + 1) x (C + 1), last row/col = background
    im_ = ax.imshow(cm / cm.sum(0, keepdims=True).clip(min=1), cmap="Blues", vmin=0, vmax=1)
    labels_ = NAMES + ["background"]
    ax.set_xticks(range(len(labels_))); ax.set_xticklabels(labels_, rotation=40, ha="right"); ax.set_yticks(range(len(labels_))); ax.set_yticklabels(labels_)
    ax.set_xlabel("true"); ax.set_ylabel("predicted"); ax.set_title(f"{name}: confusion (column-normalised)")
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j]:
                ax.text(j, i, int(cm[i, j]), ha="center", va="center", fontsize=8, color="white" if cm[i, j] > cm[:, j].sum() / 2 else "black")
plt.suptitle("Figure 4.1 - Per-class AP and confusion matrices (Ultralytics puts TRUE classes on the columns)", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"validation : mAP50 {m_val.box.map50:.3f} | mAP50-95 {m_val.box.map:.3f} | precision {m_val.box.mp:.3f} | recall {m_val.box.mr:.3f}")
print(f"dark camera: mAP50 {m_dark.box.map50:.3f} | mAP50-95 {m_dark.box.map:.3f} | precision {m_dark.box.mp:.3f} | recall {m_dark.box.mr:.3f}")

In [ ]:
def box_iou_np(a, b):
    return torchvision.ops.box_iou(torch.tensor(a, dtype=torch.float32).reshape(-1, 4), torch.tensor(b, dtype=torch.float32).reshape(-1, 4)).numpy()


def error_census(split, conf=0.25):
    """Count misses (FN), false detections (FP) and wrong classes per image, to pick images for the failure gallery."""
    out = []
    for p in sorted((ROOT / "images" / split).glob("*.jpg")):
        img = cv2.imread(str(p))
        gb, gl = read_yolo(ROOT / "labels" / split / (p.stem + ".txt"), img.shape[1], img.shape[0])
        r = best.predict(img, imgsz=IMGSZ, conf=conf, verbose=False, device=DEVICE)[0]
        pb, pl = r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int)
        iou = box_iou_np(pb, gb) if len(pb) and len(gb) else np.zeros((len(pb), len(gb)))
        if len(pb) == 0 or len(gb) == 0:                                  # nothing to match: all misses or all false detections
            out.append((len(pb) + len(gb), p, r, gb, gl))
            continue
        fn = int((iou.max(0) < 0.5).sum())
        fp = int((iou.max(1) < 0.5).sum())
        wrong = int(sum(1 for i in range(len(pb)) if iou[i].max() >= 0.5 and pl[i] != gl[iou[i].argmax()]))
        out.append((fn + fp + wrong, p, r, gb, gl))
    return sorted(out, key=lambda t: -t[0])


worst = error_census("test_dark")[:3]
fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))
for ax, (n_err, p, r, gb, gl) in zip(axes, worst):
    vis = r.plot()
    for g in gb:
        cv2.rectangle(vis, tuple(g[:2].astype(int)), tuple(g[2:].astype(int)), (255, 255, 255), 1)
    ax.imshow(vis[..., ::-1]); ax.axis("off"); ax.set_title(f"{p.name}: {n_err} errors (white = ground truth)")
plt.suptitle("Figure 4.2 - The three worst dark-camera images: where does the detector fail?", fontweight="bold")
plt.tight_layout(); plt.show()

## 5. Export to ONNX and run without PyTorch

**ONNX** (Open Neural Network Exchange) is a framework-neutral file format for the network graph and weights. One exported file runs on:

| runtime | where | notes |
|---|---|---|
| **ONNX Runtime** (Microsoft) | Python, C#, C++, Java, JS; CPU, CUDA, TensorRT, OpenVINO, DirectML providers | the reference runtime |
| **OpenCV DNN** | anywhere OpenCV runs (C++, Python, C# via OpenCvSharp) | no extra dependency |
| **TensorRT** / **OpenVINO** | NVIDIA GPUs and Jetson / Intel CPUs, iGPUs and NPUs | fastest on their hardware; FP16 / INT8 |

The exported graph contains the network only: **pre-processing (letterbox) and post-processing (decode, NMS) are your job** — exactly what you wrote in Week 8B.

In [ ]:
onnx_path = Path(best.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False))
session = ort.InferenceSession(str(onnx_path), providers=["CPUExecutionProvider"])
inp, out = session.get_inputs()[0], session.get_outputs()[0]
print(f"{onnx_path.name}: {onnx_path.stat().st_size / 1e6:.1f} MB | input {inp.name} {inp.shape} | output {out.name} {out.shape}")
print(f"{out.shape[2]} candidates = ({IMGSZ}/8)^2 + ({IMGSZ}/16)^2 + ({IMGSZ}/32)^2 ; {out.shape[1]} = 4 box numbers + {len(NAMES)} classes")


def letterbox_ref(img, size, color=114):
    h, w = img.shape[:2]
    s = size / max(h, w)
    nh, nw = int(round(h * s)), int(round(w * s))
    px, py = (size - nw) // 2, (size - nh) // 2
    canvas = np.full((size, size, 3), color, np.uint8)
    canvas[py:py + nh, px:px + nw] = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
    return canvas, s, (px, py)


def postprocess_ref(raw, s, pad, shape, conf=0.25, iou=0.7):
    scores, cls = raw[4:].max(0), raw[4:].argmax(0)
    k = scores > conf
    cx, cy, w, h = raw[:4, k]
    b = np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], 1)
    keep = torchvision.ops.batched_nms(torch.tensor(b), torch.tensor(scores[k]), torch.tensor(cls[k]), iou).numpy()
    b, sc, cl = b[keep], scores[k][keep], cls[k][keep]
    b = (b - [pad[0], pad[1], pad[0], pad[1]]) / s
    b[:, [0, 2]] = b[:, [0, 2]].clip(0, shape[1]); b[:, [1, 3]] = b[:, [1, 3]].clip(0, shape[0])
    return b, sc, cl


def ort_detect_ref(sess, img, conf=0.25):
    lb, s, pad = letterbox_ref(img, IMGSZ)
    x = lb[..., ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255.0
    raw = sess.run(None, {sess.get_inputs()[0].name: x})[0][0]
    return postprocess_ref(raw, s, pad, img.shape, conf)


test_img = cv2.imread(str(sorted((ROOT / "images" / "val").glob("*.jpg"))[0]))
b_ort, s_ort, c_ort = ort_detect_ref(session, test_img)
r_pt = best.predict(test_img, imgsz=IMGSZ, conf=0.25, verbose=False, device=DEVICE)[0]
b_pt = r_pt.boxes.xyxy.cpu().numpy()
match = box_iou_np(b_ort, b_pt).max(1) if len(b_ort) and len(b_pt) else np.array([])
print(f"PyTorch: {len(b_pt)} objects | ONNX Runtime + our post-processing: {len(b_ort)} objects | "
      f"mean IoU of matched boxes {match.mean() if len(match) else float('nan'):.4f}")
show(r_pt.plot(), partsim.draw_boxes(test_img, b_ort, c_ort, s_ort), titles=["Ultralytics (PyTorch)", "ONNX Runtime + our pre/post-processing"],
     figsize=(20, 6.5))

In [ ]:
net = cv2.dnn.readNetFromONNX(str(onnx_path))
x_bgr = letterbox_ref(test_img, IMGSZ)[0]
blob = cv2.dnn.blobFromImage(x_bgr, 1 / 255.0, (IMGSZ, IMGSZ), swapRB=True)          # the same tensor, built by OpenCV
x_np = x_bgr[..., ::-1].transpose(2, 0, 1)[None].astype(np.float32) / 255.0
runs = 3 if SMOKE else 30
timing = {}
t0 = time.perf_counter()
for _ in range(runs):
    best.predict(test_img, imgsz=IMGSZ, verbose=False, device="cpu")
timing["Ultralytics (PyTorch, CPU)"] = (time.perf_counter() - t0) / runs * 1000
t0 = time.perf_counter()
for _ in range(runs):
    session.run(None, {inp.name: x_np})
timing["ONNX Runtime (CPU)"] = (time.perf_counter() - t0) / runs * 1000
t0 = time.perf_counter()
for _ in range(runs):
    net.setInput(blob); net.forward()
timing["OpenCV DNN (CPU)"] = (time.perf_counter() - t0) / runs * 1000
net.setInput(blob)
diff = np.abs(net.forward()[0] - session.run(None, {inp.name: blob})[0][0]).max()
plt.figure(figsize=(10, 3.2)); plt.barh(list(timing), list(timing.values()), color=["#9aa0a6", "#188038", "#1a73e8"])
plt.xlabel("ms per image (network forward; Ultralytics includes pre/post-processing)")
plt.title("Figure 5.1 - The same network in three CPU runtimes", fontweight="bold"); plt.tight_layout(); plt.show()
print(f"max difference between OpenCV DNN and ONNX Runtime outputs: {diff:.2e}")

## 6. Line-rate engineering: is it fast enough?

A conveyor moves at speed $v$ (mm/s); the camera sees a field of view of length $L$ (mm) along the belt. A part is visible for $L/v$ seconds.
To see every part in at least $k$ frames (for tracking and voting, Week 10), the camera and the whole pipeline must run at

$$f_\text{required} = \frac{k\,v}{L} \quad \text{frames per second.}$$

The **latency budget** per frame is $1/f$: acquisition + pre-processing + inference + post-processing + communication (PLC, robot) must fit in it,
with margin (plan for ≤ 70% of the budget).

In [ ]:
v, L, k = 2000.0, 320.0, 3              # 2 m/s belt, 320 mm field of view (640 px at 0.5 mm/px), see each part 3 times
f_req = k * v / L
budget = 1000 / f_req
stages = {"acquire + transfer": 8.0, "letterbox + tensor": 2.0, "inference (ONNX Runtime)": timing["ONNX Runtime (CPU)"],
          "decode + NMS": 1.5, "send result to PLC": 2.0}
fig, ax = plt.subplots(figsize=(14, 3.2))
left = 0
for (name, ms), col in zip(stages.items(), ["#9aa0a6", "#fbbc05", "#34a853", "#4285f4", "#ea4335"]):
    ax.barh(0, ms, left=left, color=col, edgecolor="white", label=f"{name}: {ms:.1f} ms")
    left += ms
ax.axvline(budget, color="k", lw=2); ax.text(budget, 0.45, f" budget {budget:.0f} ms ({f_req:.1f} FPS)", va="center")
ax.axvline(0.7 * budget, color="k", ls="--", label="70% of the budget"); ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.35), ncol=3, fontsize=8); ax.set_yticks([]); ax.set_xlim(0, max(budget, left) * 1.15); ax.set_xlabel("ms")
ax.set_title(f"Figure 6.1 - Latency budget: {left:.0f} ms used of {budget:.0f} ms ({left / budget:.0%})", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. The C# corner (optional, run locally)

Many factory applications (HMIs, inspection stations, PLC gateways) are written in **C#**. The *same* `best.onnx` runs there with the
`Microsoft.ML.OnnxRuntime` NuGet package; images are handled with **OpenCvSharp**. The program in `resources/csharp/YoloOnnx/` repeats every step of
Section 5 (letterbox → tensor → run → decode → undo letterbox → NMS). Continuous integration checks that it builds; run it on your laptop:

```bash
cd resources/csharp/YoloOnnx
dotnet run -- ../../../data/w09b/best.onnx ../../../data/w09b/scene.jpg 0.25
```

The next cell copies the model and a test scene to `data/w09b/` for that command and prints the C# source.

In [ ]:
shutil.copy(onnx_path, ROOT / "best.onnx")
cv2.imwrite(str(ROOT / "scene.jpg"), test_img)
cs = Path("resources/csharp/YoloOnnx/Program.cs")
print(cs.read_text() if cs.exists() else "resources/csharp/YoloOnnx/Program.cs not found (clone the full repository).")

## 8. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. YOLO label lines
Return a list of strings `"class cx cy w h"` (normalised, **6 decimals**) for xyxy pixel boxes, as written in Section 2. Do not call `yolo_lines_ref`.

In [ ]:
def yolo_lines(boxes, labels, img_w, img_h):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    lines = yolo_lines(np.array([[0, 0, 64, 48], [320, 240, 640, 480]]), [2, 0], 640, 480)
    assert lines == ["2 0.050000 0.050000 0.100000 0.100000", "0 0.750000 0.750000 0.500000 0.500000"], f"got {lines}"
    img, b, l = partsim.make_scene(5)
    assert yolo_lines(b, l, img.shape[1], img.shape[0]) == yolo_lines_ref(b, l, img.shape[1], img.shape[0]), "differs from the reference on a scene"


check("E1 yolo_lines", _test_e1)

### E2. Required frame rate
Return the frame rate (FPS) needed to see every part in at least `views` frames, for a belt speed `v_mm_s` and a field of view `fov_mm` along the belt (Section 6).

In [ ]:
def required_fps(v_mm_s, fov_mm, views=3):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    assert abs(required_fps(500, 320, 3) - 4.6875) < 1e-9, "500 mm/s, 320 mm, 3 views -> 4.6875 FPS"
    assert abs(required_fps(2000, 100, 2) - 40) < 1e-9, "2 m/s, 100 mm, 2 views -> 40 FPS"
    assert required_fps(1000, 200, 6) > required_fps(1000, 200, 3), "more views need more FPS"


check("E2 required_fps", _test_e2)

### E3. ONNX Runtime detector
Write `ort_detect(sess, img, conf=0.25)` that returns `(boxes_xyxy, scores, classes)` in original-image pixels, using `letterbox_ref` and `postprocess_ref` from Section 5 (RGB, CHW, float32 in [0, 1]). Do not call `ort_detect_ref`.

In [ ]:
def ort_detect(sess, img, conf=0.25):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    img = cv2.imread(str(sorted((ROOT / "images" / "val").glob("*.jpg"))[1]))
    b, s, c = [np.asarray(v) for v in ort_detect(session, img)]
    rb, rs, rc = ort_detect_ref(session, img)
    assert b.shape == rb.shape, f"expected {rb.shape} boxes, got {b.shape}"
    assert np.allclose(b, rb, atol=1e-3) and np.array_equal(c, rc), "differs from the reference pipeline"


check("E3 ort_detect", _test_e3)

### E4. Robustness through data (no self-check)
The detector was trained with exposures 0.6–1.1. Retrain with exposures 0.4–1.2 and sensor noise up to 8, and compare the dark-camera mAP50-95.
Then add `degrees=180` (rotation augmentation) and compare mAP50 and mAP50-95. Which change helped or hurt, and why? (Hint: what happens to an axis-aligned box when the image is rotated by 45°?) (Keep everything else fixed: this is Week 9A's method.)

In [ ]:
# E4: your code here

## 9. Challenge: from notebook to line
1. Make a 30-second "conveyor video" by translating a long scene (e.g. `partsim.make_scene(size=(480, 2000))`) under a 480 × 640 window, run the
   ONNX detector on every frame, and measure the end-to-end FPS on your laptop.
2. Export to **OpenVINO** (`format="openvino"`) or **TensorRT** (`format="engine"`, NVIDIA GPU) and compare speed and mAP with ONNX Runtime.
3. Try **INT8** quantisation (OpenVINO or TensorRT with a calibration set). What happens to the smallest class, `hex_nut`?

In [ ]:
# Your challenge code here

## 10. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'What does one line of a YOLO label file contain?',
        "options": [
            'x_min y_min x_max y_max in pixels',
            'class cx cy w h, normalised to 0-1',
            'class x y w h in pixels',
            'a polygon',
        ],
        "answer": '2b820165b7',
        "explain": 'One line per object, one .txt per image.',
    },
    {
        "q": 'Why fine-tune from COCO-pretrained weights instead of training from scratch?',
        "options": [
            'It is required by YOLO',
            'The pretrained features transfer, so fewer images and epochs are needed',
            'It removes the need for labels',
            'It makes inference faster',
        ],
        "answer": '5efb84e346',
        "explain": 'Week 6B: transfer learning.',
    },
    {
        "q": "Why keep a separate 'dark camera' test set that is never used for training or model selection?",
        "options": [
            'To make training faster',
            'To measure robustness honestly',
            'To increase mAP',
            'Because YOLO needs three splits',
        ],
        "answer": 'ebc63d0bee',
        "explain": 'Anything used for decisions is no longer an unbiased test.',
    },
    {
        "q": 'What is NOT inside an exported YOLO ONNX file?',
        "options": [
            'The convolution weights',
            'The network graph',
            'The letterbox pre-processing and NMS',
            'The input shape',
        ],
        "answer": '12ab8139b3',
        "explain": 'Pre/post-processing must be re-implemented in the target language.',
    },
    {
        "q": 'A belt moves at 1 m/s, the field of view is 250 mm and each part must be seen 2 times. Required FPS?',
        "options": [
            '2',
            '4',
            '8',
            '16',
        ],
        "answer": 'd3fdfa2c02',
        "explain": 'f = k v / L = 2 * 1000 / 250 = 8.',
    },
    {
        "q": 'Which runtime lets the same .onnx file run inside a C# application?',
        "options": [
            'TensorBoard',
            'ONNX Runtime',
            'Matplotlib',
            'pandas',
        ],
        "answer": 'bfdebe44de',
        "explain": 'Via the Microsoft.ML.OnnxRuntime NuGet package (OpenCV DNN via OpenCvSharp also works).',
    },
    {
        "q": 'In a latency budget, why plan to use at most ~70% of the frame time?',
        "options": [
            'To save energy',
            'To leave margin for jitter, OS scheduling and bursts',
            'Because GPUs are slow',
            'It is required by ONNX',
        ],
        "answer": '5079254583',
        "explain": 'Real-time systems need headroom.',
    },
], title='Week 09B quiz')

## 11. Exit ticket (reflection)
1. Which step of today's pipeline would take the longest in a real project, and why?
2. The dark-camera mAP is lower than validation. List two data changes and one deployment change that could close the gap.
3. Which runtime would you choose for your project's target hardware, and what would you have to re-implement there?

## 12. What's new (2025–26)

> - **Export targets keep multiplying**: ONNX, TensorRT, OpenVINO, CoreML, TFLite/LiteRT, NCNN, RKNN (Rockchip NPUs), Hailo and IMX500 smart
>   cameras; Ultralytics, Hugging Face Optimum and NVIDIA Model Optimizer automate FP16/INT8 conversion.
> - **NMS-free models** (YOLO26, YOLOv10, RT-DETR/D-FINE) make exported graphs end-to-end, so the post-processing in C# or on a microcontroller
>   becomes a simple threshold.
> - **ONNX Runtime GenAI and Windows ML** bring the same ONNX files to NPUs in laptops; `onnxruntime-web` runs them in the browser (WebGPU).
> - **MLOps for vision**: dataset versioning, model registries and drift monitoring (FiftyOne, Roboflow, ClearML, MLflow) connect this session to
>   Week 9A's data flywheel.

## 13. References and further exploration
- Ultralytics documentation: *Train*, *Val*, *Export*, *Dataset formats (YOLO)*, *Hyper-parameter tuning*. [docs.ultralytics.com](https://docs.ultralytics.com/)
- ONNX: *Open Neural Network Exchange* specification. [onnx.ai](https://onnx.ai/) · ONNX Runtime docs (Python, C#, execution providers). [onnxruntime.ai/docs](https://onnxruntime.ai/docs/)
- OpenCV DNN module tutorials. [docs.opencv.org](https://docs.opencv.org/4.x/d2/d58/tutorial_table_of_content_dnn.html)
- OpenCvSharp (C# wrapper for OpenCV). [github.com/shimat/opencvsharp](https://github.com/shimat/opencvsharp)
- Jacob, B. et al. (2018). *Quantization and training of neural networks for efficient integer-arithmetic-only inference.* CVPR. [arXiv:1712.05877](https://arxiv.org/abs/1712.05877)
- NVIDIA TensorRT and Intel OpenVINO developer guides. [developer.nvidia.com/tensorrt](https://developer.nvidia.com/tensorrt) · [docs.openvino.ai](https://docs.openvino.ai/)
- Sculley, D. et al. (2015). *Hidden technical debt in machine learning systems.* NeurIPS. [NeurIPS proceedings](https://papers.nips.cc/paper/5656-hidden-technical-debt-in-machine-learning-systems)

**Data:** all images are generated by `resources/scripts/partsim.py`; YOLO11n COCO weights from Ultralytics (AGPL-3.0).

## 14. Key takeaways
- A detection dataset = images + one label file per image + `data.yaml`; **draw labels back** before training.
- **Fine-tune** a pretrained model; set `imgsz`, `batch`, `epochs` for your hardware; read the curves and `train_batch0.jpg`.
- Evaluate per class and on a **held-out shifted set**; look at the worst images.
- **ONNX** decouples training from deployment; you own the **pre/post-processing** in every runtime (Python, OpenCV, C#).
- Size the system with the **required FPS** and a **latency budget** with margin.

**Next week (10A):** motion — frame differencing, background subtraction and optical flow, then tracking the objects this detector finds (10B).